## Day 12 predictions (written before training)

1. **Overall val macro-F1:** ≈ 0.72 (vs LR 0.694, LSTM 0.674) → a modest win of about +0.025.
   Most classes are keyword-driven, so LR is already close to the ceiling there; the gain must come from the hard/small classes.

2. **Debt or credit management F1:** ≈ 0.35 (vs LR 0.275, LSTM 0.000).
   Pretraining already knows "settlement", "negotiate" and "relief" are related, so it shouldn't ignore the class completely like the LSTM did.
   But it's still only 0.25% of tickets with no class weights, and it overlaps with Debt collection, so I don't expect a big fix.

3. **Biggest gain over LR:** Debt or credit management, because it's defined by meaning
   (someone *helping* me with debt) rather than one unique keyword, which is exactly what pretrained embeddings add.

4. **Decision rule:** deploy the Transformer only if it
   - beats LR by **> 0.01 macro-F1** (and the gain holds on the test set on Day 13),
   - **fits in 512 MB** via ONNX,
   - answers in **< 200 ms per ticket** on Render,
   - and **doesn't score lower than LR on any small class** (≤ 1,300 tickets).
   Otherwise LR stays.


In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.text import clean_text

train = pd.read_parquet("../data/processed/train.parquet")
train["text_clean"] = train["Consumer complaint narrative"].map(clean_text)

def first_350(text: str) -> str:
    return " ".join(text.split()[:350])

train["text_350"] = train["text_clean"].map(first_350)


In [4]:
import kagglehub, keras_hub

local = kagglehub.model_download("keras/bert/keras/bert_small_en_uncased/3")   # ~120 MB, once
print(local)
tokenizer = keras_hub.models.BertTokenizer.from_preset(local)
print(tokenizer("not my account"))



100%|██████████| 208/208 [00:00<00:00, 117kB/s]


Download already complete (455 bytes).


100%|██████████| 226k/226k [00:01<00:00, 173kB/s]


Download already complete (761 bytes).


100%|██████████| 110M/110M [01:13<00:00, 1.58MB/s]

C:\Users\srisu\.cache\kagglehub\models\keras\bert\keras\bert_small_en_uncased\3
[2025 2026 4070]


In [5]:
texts = train["text_350"].sample(1000, random_state=42).tolist()

n_tokens = np.array([len(tokenizer(t)) + 2 for t in texts])   # + [CLS] and [SEP]
n_words  = np.array([len(t.split()) for t in texts])

print("median:", np.median(n_tokens))
print("95th pct:", np.percentile(n_tokens, 95))
print("max:", n_tokens.max())
print(f"% above 512: {(n_tokens > 512).mean():.1%}")
print(f"tokens per word: {n_tokens.sum() / n_words.sum():.2f}")


median: 237.5
95th pct: 634.1499999999999
max: 1592
% above 512: 10.1%
tokens per word: 1.60


In [6]:
for w in ["redactedinfo", "redacteddate", "redactedamount"]:
    print(w, "→", tokenizer(w), len(tokenizer(w)))

for L in [256, 384, 512]:
    print(f"cut at {L}: {(n_tokens > L).mean():.1%}")


redactedinfo → [ 2417 18908  2098  2378 14876] 5
redacteddate → [ 2417 18908 22367  3686] 4
redactedamount → [ 2417 18908 11960 20048] 4
cut at 256: 47.2%
cut at 384: 28.1%
cut at 512: 10.1%


In [8]:
candidates = ["censored", "omitted", "withheld", "concealed", "masked",
              "anonymous", "classified", "blank", "unknown", "erased"]

for w in candidates:
    ids = tokenizer(w)
    if len(ids) == 1:                                    # only single-token words
        n = train["text_clean"].str.contains(rf"\b{w}\b").sum()
        print(f"{w:12} → {ids}  | in {n} train complaints")


omitted      → [16647]  | in 178 train complaints
concealed    → [14091]  | in 57 train complaints
masked       → [16520]  | in 31 train complaints
anonymous    → [10812]  | in 55 train complaints
classified   → [6219]  | in 477 train complaints
blank        → [8744]  | in 465 train complaints
unknown      → [4242]  | in 5252 train complaints
erased       → [23516]  | in 495 train complaints


In [ ]:
import importlib, src.text; importlib.reload(src.text)   # pick up the new function
from src.text import format_for_bert

n_tok2 = np.array([len(tokenizer(format_for_bert(t))) + 2 for t in texts])
print("median:", np.median(n_tok2), "| 95th:", np.percentile(n_tok2, 95), "| max:", n_tok2.max())
for L in [256, 384, 512]:
    print(f"cut at {L}: {(n_tok2 > L).mean():.1%}   (before: {(n_tokens > L).mean():.1%})")


median: 181.0 | 95th: 416.0 | max: 465
cut at 256: 33.7%   (before: 47.2%)
cut at 384: 14.1%   (before: 28.1%)
cut at 512: 0.0%   (before: 10.1%)


: 